# Rumus model skrining 5 butir — hs-CRP IFLS-5 (untuk model card naskah HISS)

Notebook ini membuat **rumus siap pakai** dari model logistik 5 butir (IMT, jenis kelamin, umur, kesehatan diri buruk, tinggal di kota):
koefisien + intercept dalam skala asli, kalibrasi ulang Platt, ambang rujukan, dan rumus Excel.

Pembagian data latih/uji dan model **identik** dengan `analisis_model_sederhana.ipynb` (seed 42).
Validasi silang berulang yang lama **tidak** dijalankan di sini, jadi notebook ini selesai dalam 1–2 menit.
Hasil disimpan di `results/rumus_model_lima_butir.xlsx`.

In [ ]:
# === SEL 0: folder kerja dan file data ===
import os
ROOT = r"F:\SINTA 3\crp-ifls5-ml"
OUTDIR = os.path.join(ROOT, "results")
os.makedirs(OUTDIR, exist_ok=True)
os.chdir(OUTDIR)
DATA_FILE = os.path.join(ROOT, "data", "crp_ifls5_model.csv")
ANALYTIC_FILE = r"F:\SINTA 3\olahan_ifls5\ifls5_analytic.csv"   # untuk analisis CRP > 10 dan perbandingan peserta dikeluarkan
assert os.path.exists(DATA_FILE), f"Data tidak ditemukan: {DATA_FILE}"
print("Folder kerja :", os.getcwd())
print("Data model   :", DATA_FILE)
print("Data analitik:", ANALYTIC_FILE, "(ada)" if os.path.exists(ANALYTIC_FILE) else "(TIDAK ADA - analisis 4a dan 4c dilewati)")

# Jumlah ulangan validasi silang. 5 = cukup untuk naskah (sekitar 30-60 menit dengan DNN). Naikkan ke 10 bila ada waktu.
REPEATS = 5
RUN_DNN = False   # DNN tidak diperlukan untuk perbandingan ini

In [ ]:
import warnings, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
from scipy import stats
from sklearn.model_selection import (train_test_split, StratifiedKFold, StratifiedGroupKFold, RepeatedStratifiedKFold,
                                     cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.calibration import calibration_curve
from sklearn.base import clone, BaseEstimator, ClassifierMixin

def ada(m):
    try: __import__(m); return True
    except ImportError: return False
HAS_XGB, HAS_LGBM, HAS_TORCH = ada('xgboost'), ada('lightgbm'), ada('torch')
MODE = 'FINAL' if (HAS_XGB and HAS_LGBM) else 'PRATINJAU'
print(f'XGBoost={HAS_XGB} | LightGBM={HAS_LGBM} | PyTorch={HAS_TORCH} | MODE: {MODE}')
SEED = 42

In [ ]:
df = pd.read_csv(DATA_FILE, dtype={'pidlink': str, 'hhid14': str})
NUM = ['age', 'lnpce', 'hhsize', 'cig_day', 'pa_vig_days', 'pa_mod_days', 'pa_walk_days', 'bmi', 'sbp', 'dbp', 'n_chronic', 'cesd10']
BIN = ['female', 'married', 'widowed', 'urban', 'dx_hypert', 'dx_diab', 'dx_heart', 'dx_stroke', 'dx_chol',
       'dx_asthma', 'dx_lung', 'dx_arthritis', 'dx_kidney', 'dx_liver', 'dx_digest', 'med_hypert', 'med_diab', 'srh_poor']
CAT = ['educ', 'smoke_status']
FEATS = NUM + BIN + CAT
X, y, groups = df[FEATS].copy(), df.crp_high.values, df.hhid14.values

# Pembagian latih/uji IDENTIK dengan notebook utama (seed 42, stratified)
X_tr, X_te, y_tr, y_te, idx_tr, idx_te = train_test_split(X, y, df.index, test_size=0.2, stratify=y, random_state=SEED)
hh_tr, hh_te = set(df.loc[idx_tr, 'hhid14']), df.loc[idx_te, 'hhid14']
shared = hh_te.isin(hh_tr)
n_hh = df.hhid14.nunique(); size = df.groupby('hhid14').size()
print(f'n = {len(df)}, kasus = {y.sum()} ({y.mean()*100:.1f}%) | rumah tangga = {n_hh}; '
      f'{(size > 1).sum()} rumah tangga punya >1 peserta ({size[size > 1].sum()} peserta)')
print(f'Data uji: {shared.sum()} dari {len(idx_te)} peserta ({shared.mean()*100:.1f}%) punya anggota rumah tangga di data latih')

def prep(scale, feats=FEATS):
    num, bin_, cat = [c for c in NUM if c in feats], [c for c in BIN if c in feats], [c for c in CAT if c in feats]
    num_steps = [('imp', SimpleImputer(strategy='median'))] + ([('sc', StandardScaler())] if scale else [])
    return ColumnTransformer([('num', Pipeline(num_steps), num), ('bin', SimpleImputer(strategy='most_frequent'), bin_),
                              ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                                                ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat)],
                             verbose_feature_names_out=False)

In [ ]:
# === Model dengan hyperparameter terbaik dari notebook utama ===
class BalancedMLP(MLPClassifier):
    def fit(self, X, y, sample_weight=None):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y))

if HAS_TORCH:
    import torch, torch.nn as nn
    class TorchDNN(ClassifierMixin, BaseEstimator):
        def __init__(self, hidden=(64, 32), dropout=0.3, lr=1e-3, weight_decay=1e-5, epochs=300, batch=128, patience=25, seed=SEED):
            self.hidden, self.dropout, self.lr, self.weight_decay = hidden, dropout, lr, weight_decay
            self.epochs, self.batch, self.patience, self.seed = epochs, batch, patience, seed
        def _net(self, d):
            layers = []
            for h in self.hidden: layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(self.dropout)]; d = h
            return nn.Sequential(*layers, nn.Linear(d, 1))
        def fit(self, X, y):
            torch.manual_seed(self.seed); X = np.asarray(X, np.float32); y = np.asarray(y, np.float32)
            self.classes_ = np.array([0, 1])
            Xa, Xv, ya, yv = train_test_split(X, y, test_size=0.15, stratify=y, random_state=self.seed)
            self.net_ = self._net(X.shape[1]); opt = torch.optim.AdamW(self.net_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float((ya == 0).sum() / (ya == 1).sum())))
            Xa_t, ya_t, Xv_t = torch.tensor(Xa), torch.tensor(ya), torch.tensor(Xv)
            best, wait, state = -1, 0, None
            for ep in range(self.epochs):
                self.net_.train(); perm = torch.randperm(len(Xa_t))
                for i in range(0, len(perm), self.batch):
                    b = perm[i:i + self.batch]
                    if len(b) < 2: continue
                    opt.zero_grad(); lossf(self.net_(Xa_t[b]).squeeze(1), ya_t[b]).backward(); opt.step()
                self.net_.eval()
                with torch.no_grad(): auc = roc_auc_score(yv, self.net_(Xv_t).squeeze(1).numpy())
                if auc > best: best, wait, state = auc, 0, {k: v.clone() for k, v in self.net_.state_dict().items()}
                else:
                    wait += 1
                    if wait >= self.patience: break
            self.net_.load_state_dict(state); return self
        def predict_proba(self, X):
            self.net_.eval()
            with torch.no_grad(): p = torch.sigmoid(self.net_(torch.tensor(np.asarray(X, np.float32))).squeeze(1)).numpy()
            return np.c_[1 - p, p]
        def predict(self, X): return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

POS_W = (y_tr == 0).sum() / (y_tr == 1).sum()      # sama dengan notebook utama (5,62)
if HAS_XGB: from xgboost import XGBClassifier

def make_models(feats=FEATS):
    M = {}
    M['Logistic regression'] = Pipeline([('p', prep(True, feats)), ('m', LogisticRegression(C=0.011288378916846888, class_weight='balanced', max_iter=3000))])
    M['Random forest'] = Pipeline([('p', prep(False, feats)), ('m', RandomForestClassifier(n_estimators=500, max_depth=8, min_samples_leaf=40,
                                   max_features='sqrt', class_weight='balanced_subsample', n_jobs=-1, random_state=SEED))])
    if HAS_XGB:
        M['XGBoost'] = Pipeline([('p', prep(False, feats)), ('m', XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.01, subsample=0.85, colsample_bytree=0.8,
                                 min_child_weight=10, scale_pos_weight=POS_W, eval_metric='logloss', n_jobs=-1, random_state=SEED))])
    else:
        M['HistGB (pengganti XGBoost)'] = Pipeline([('p', prep(False, feats)), ('m', HistGradientBoostingClassifier(class_weight='balanced', max_iter=200,
                                                   learning_rate=0.03, max_leaf_nodes=7, random_state=SEED))])
    if HAS_LGBM:
        from lightgbm import LGBMClassifier
        M['LightGBM'] = Pipeline([('p', prep(False, feats)), ('m', LGBMClassifier(n_estimators=200, num_leaves=15, learning_rate=0.01, min_child_samples=50,
                                  subsample=0.7, colsample_bytree=0.6, class_weight='balanced', random_state=SEED, verbose=-1))])
    M['MLP'] = Pipeline([('p', prep(True, feats)), ('m', BalancedMLP(hidden_layer_sizes=(64, 32), alpha=1.0, learning_rate_init=0.003, batch_size=64,
                         early_stopping=True, validation_fraction=0.15, n_iter_no_change=20, max_iter=600, random_state=SEED))])
    if HAS_TORCH and RUN_DNN:
        M['DNN (PyTorch)'] = Pipeline([('p', prep(True, feats)), ('m', TorchDNN())])
    M['Ensemble (RF + MLP + LR)'] = VotingClassifier([('rf', clone(M['Random forest'])), ('mlp', clone(M['MLP'])),
                                                      ('lr', clone(M['Logistic regression']))], voting='soft')
    return M

MODELS = make_models()
print('Model:', list(MODELS))

In [ ]:
SIMPLE = {'LR: BMI only': ['bmi'],
          'LR: 5 items (BMI, sex, age, self-rated health, urban)': ['bmi', 'female', 'age', 'srh_poor', 'urban']}
def simple_model():
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                     ('m', LogisticRegression(class_weight='balanced', max_iter=2000))])

# Model pembanding dari notebook utama (dengan hyperparameter yang sama)
REF_MODELS = {k: v for k, v in MODELS.items() if k in ('Ensemble (RF + MLP + LR)', 'XGBoost', 'HistGB (pengganti XGBoost)', 'Logistic regression')}
print('Pembanding:', list(REF_MODELS), '| Model sederhana:', list(SIMPLE))

def sens_at_spec(yt, p, spec):
    f, t, _ = roc_curve(yt, p); return t[f <= 1 - spec].max()

## 1. Model 5 butir dan koefisien skala asli

In [ ]:
cv5 = StratifiedKFold(5, shuffle=True, random_state=SEED)
cols = SIMPLE['LR: 5 items (BMI, sex, age, self-rated health, urban)']
print('Kolom:', cols)

# Model akhir (imputasi median -> standarisasi -> LR class_weight balanced), sama dengan naskah
m5 = simple_model().fit(X_tr[cols], y_tr)
imp, sc, lr = m5.named_steps['imp'], m5.named_steps['sc'], m5.named_steps['m']

# Ubah koefisien terstandar ke skala asli
mean, sd = sc.mean_, sc.scale_
b_std, b0_std = lr.coef_[0], lr.intercept_[0]
b_raw = b_std / sd
b0_raw = b0_std - np.sum(b_std * mean / sd)
coef5 = pd.DataFrame({'variable': cols, 'mean_train': mean, 'sd_train': sd,
                      'coef_standardised': b_std, 'coef_raw_scale': b_raw,
                      'OR_per_unit_raw': np.exp(b_raw)})
coef5.loc[len(coef5)] = ['(intercept)', np.nan, np.nan, b0_std, b0_raw, np.nan]
display(coef5.round(4))

# Cek: rumus manual harus sama dengan sklearn
lp_manual = b0_raw + X_tr[cols].fillna(pd.Series(imp.statistics_, index=cols)).values @ b_raw
assert np.allclose(1/(1+np.exp(-lp_manual)), m5.predict_proba(X_tr[cols])[:, 1]), 'rumus tidak cocok'
print('Cek rumus manual = sklearn: OK')

## 2. Kalibrasi ulang (Platt) dan ambang rujukan

In [ ]:
# Prediksi out-of-fold di data latih (class_weight membuat risiko terlalu tinggi, jadi perlu kalibrasi ulang)
oof = cross_val_predict(simple_model(), X_tr[cols], y_tr, cv=cv5, method='predict_proba')[:, 1]
lp_oof = np.log(oof / (1 - oof))
platt = LogisticRegression(C=1e6, max_iter=1000).fit(lp_oof.reshape(-1, 1), y_tr)
a, b = platt.intercept_[0], platt.coef_[0][0]
print(f'Platt: p_kalibrasi = 1 / (1 + exp(-({a:.4f} + {b:.4f} * LP)))')

# Ambang Youden pada prediksi out-of-fold (sama dengan naskah)
f, t, th = roc_curve(y_tr, oof); thr = th[np.argmax(t - f)]
thr_lp = np.log(thr / (1 - thr))
thr_cal = 1 / (1 + np.exp(-(a + b * thr_lp)))
print(f'Ambang: probabilitas (belum kalibrasi) {thr:.4f} | LP {thr_lp:.4f} | probabilitas terkalibrasi {thr_cal:.4f}')

## 3. Cek di data uji (harus sama dengan naskah: AUC 0.724, sensitivitas 0.646, spesifisitas 0.704)

In [ ]:
p_te = m5.predict_proba(X_te[cols])[:, 1]; yhat = (p_te >= thr).astype(int)
sens = ((yhat == 1) & (y_te == 1)).sum() / (y_te == 1).sum()
spec = ((yhat == 0) & (y_te == 0)).sum() / (y_te == 0).sum()
auc5 = roc_auc_score(y_te, p_te)
print(f'Data uji: AUC {auc5:.3f} | sensitivitas {sens:.3f} | spesifisitas {spec:.3f}')

## 4. Rumus Excel dan simpan hasil

Kolom Excel: A = IMT (kg/m²), B = perempuan (1/0), C = umur (tahun), D = kesehatan diri buruk (1/0), E = tinggal di kota (1/0).

In [ ]:
names = {'bmi': 'A2', 'female': 'B2', 'age': 'C2', 'srh_poor': 'D2', 'urban': 'E2'}
lp_excel = f'{b0_raw:.6f}' + ''.join(f'{"+" if v >= 0 else "-"}{abs(v):.6f}*{names[c]}' for c, v in zip(cols, b_raw))
excel = pd.DataFrame({'item': ['LP (sel F2)', 'Probabilitas terkalibrasi (sel G2)', 'Flag rujukan (sel H2)'],
                      'rumus_excel': [f'={lp_excel}', f'=1/(1+EXP(-({a:.6f}+{b:.6f}*F2)))', f'=IF(F2>={thr_lp:.6f},1,0)']})
pd.set_option('display.max_colwidth', None)
display(excel)

param = pd.DataFrame({'parameter': ['platt_a', 'platt_b', 'threshold_prob_uncal', 'threshold_LP', 'threshold_prob_calibrated',
                                    'test_AUC', 'test_sens', 'test_spec', 'imputer_median'],
                      'nilai': [a, b, thr, thr_lp, thr_cal, auc5, sens, spec,
                                ', '.join(f'{c}={v:.3f}' for c, v in zip(cols, imp.statistics_))]})
with pd.ExcelWriter('rumus_model_lima_butir.xlsx') as xw:
    coef5.round(6).to_excel(xw, sheet_name='koefisien', index=False)
    param.to_excel(xw, sheet_name='kalibrasi_ambang', index=False)
    excel.to_excel(xw, sheet_name='rumus_excel', index=False)
print('Tersimpan:', os.path.join(os.getcwd(), 'rumus_model_lima_butir.xlsx'))